# 🗂️ ISOM 260: Knowledge Agents — Retrieval-Augmented Generation (RAG)

**Session 7 — Knowledge Agents — RAG** | Suffolk University | Prof. Hasan Arslan

---

## From Session 6 to Session 7: Your Agent Learns to Read

Your agent from Session 6 can **reason**, **search Wikipedia**, and **fetch web pages**. It follows the ReAct pattern — think before acting, evaluate before concluding.

But ask it about **YOUR company's** refund policy, pricing tiers, or HR guidelines — it **hallucinates confidently**. It makes up numbers, invents policies, and presents fiction as fact.

**RAG fixes this:** search your documents FIRST, then generate answers **grounded in real data**.

This is the **#1 pattern in enterprise AI**. Every company deploying AI agents uses some form of RAG.

```
AI Agent = LLM + Tools + ReAct Reasoning + 📄 Company Knowledge (NEW!)
```

Today you'll build a **knowledge agent** that answers questions about a company using **only** its own documents — and says "I don't know" when the documents don't have the answer.

---

## 🚀 Part 1: Setup (2 minutes)

Same setup as Sessions 4, 5, and 6 — install the SDK and connect your API key.

In [ ]:
# Install dependencies
# anthropic = Claude SDK (same as Sessions 4-6)
# requests + beautifulsoup4 = for real API tools (same as Sessions 5-6)
!pip install -q anthropic requests beautifulsoup4

In [ ]:
# Set your API key
# Same key you've been using since Session 4 — it still works!
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named ANTHROPIC_API_KEY
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Option 2: Paste directly (less secure, but works for class)
    os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # <-- Replace this
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

In [ ]:
# Verify connection
import anthropic
import json

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'Session 7 connected!' and nothing else."}]
)

print(f"🟢 {response.content[0].text}")
print(f"   Model: {response.model}")

---

## 🔍 Part 2: The Knowledge Problem (Demo)

Let's see the problem RAG solves. We'll tell Claude it **IS** CloudSync's support agent — a fictional company it has never seen documents for. Watch it role-play confidently and **hallucinate an entire policy**.

CloudSync doesn't exist. Claude has never seen its documents. But with a role-play prompt, it will commit to the character and invent detailed, convincing answers...

In [ ]:
# ============================================
# THE KNOWLEDGE PROBLEM: Claude hallucinates about YOUR company
# ============================================
# CloudSync Solutions is a FICTIONAL company. Claude has never seen its docs.
# But watch — it will role-play as their support agent and make up an entire
# refund policy with specific numbers, emails, and timelines.
# The role-play prompt makes the hallucination MORE convincing, not less.

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=700,
    system=(
        "You are CloudSync Solutions' customer support assistant. "
        "You help customers with their questions about CloudSync products, "
        "policies, and services. Answer confidently and helpfully with "
        "specific details about CloudSync's offerings."
    ),
    messages=[{"role": "user", "content": 
        "What is CloudSync Solutions' refund policy for annual plans? "
        "Be specific about timelines, conditions, and how to request a refund. "
        "Include the support email and any important exceptions."
    }]
)

print("❌ THE HALLUCINATION PROBLEM")
print("=" * 60)
print(response.content[0].text)
print("\n" + "=" * 60)
print("⚠️  NONE of this is real! Claude made it all up.")
print("   It sounds authoritative, but every detail is fabricated.")
print("   The role-play prompt made it commit to the character —")
print("   producing MORE convincing fiction, not less.")
print("   This is the problem RAG solves.")

### 🔴 Hallucination Autopsy

Look at the answer above. The model confidently generated specific details about a company that **does not exist**. Let's catalog the kinds of errors it typically produces:

| Claim Made (typical hallucination) | Reality (from actual docs we'll load next) | Source |
|---|---|---|
| "60-day refund window" or "14-day refund window" | Actually **30-day** money-back guarantee for annual plans | refund-cancellation-v3.pdf |
| "Full refund at any time" or "pro-rated refund for annual" | Full refund only within first 30 days; **no refund** after that for annual plans | refund-cancellation-v3.pdf |
| "Contact billing@cloudsync.com" or made-up email | Correct email is **support@cloudsync.com** with subject line "REFUND REQUEST" | refund-cancellation-v3.pdf |
| Invented exceptions or conditions | Enterprise contracts have separate rules (non-refundable after onboarding) | refund-cancellation-v3.pdf |
| Made-up processing times | Actual processing: **5-10 business days** to original payment method | refund-cancellation-v3.pdf |

**The danger:** Every claim SOUNDS plausible. A customer reading this would believe it. This is why RAG matters — without document grounding, the agent fills gaps with plausible-sounding fiction.

> **Compare your output above** to this table. How many details did the model get wrong? Probably ALL of them — because CloudSync Solutions doesn't exist, so there's nothing to get "right" without documents.

---

## 📄 Part 3: Your Company's Documents

Let's create the **real** documents for our fictional company, **CloudSync Solutions** (a B2B SaaS platform).

In a real deployment, these would come from:
- Internal wikis (Confluence, Notion)
- Policy PDFs
- Product documentation
- HR handbooks
- Customer FAQ pages

For class, we'll define them as Python dictionaries with `title`, `content`, and `source` fields.

In [ ]:
# ============================================
# CLOUDSYNC SOLUTIONS — Company Document Library
# ============================================
# 7 documents covering policies, pricing, support, security, HR, and FAQ.
# Document 7 is DELIBERATELY outdated — we'll use it to test failure modes later.

company_documents = [
    {
        "title": "Refund & Cancellation Policy",
        "source": "policy/refund-cancellation-v3.pdf",
        "last_updated": "March 2024",
        "content": (
            "CloudSync Solutions Refund & Cancellation Policy (Effective March 2024)\n\n"
            "1. Annual Plans: 30-day money-back guarantee from the date of purchase. "
            "Full refund issued if cancellation is requested within the first 30 days. "
            "After 30 days, no refunds are available for the remainder of the annual term.\n\n"
            "2. Monthly Plans: Pro-rated refunds are available if cancellation is requested "
            "within the first 14 days of any billing cycle. After 14 days, the current "
            "month's charge is non-refundable, but no future charges will be made.\n\n"
            "3. Enterprise Contracts: Custom enterprise agreements are non-refundable after "
            "the onboarding process has been completed (typically 2-3 weeks after signing). "
            "Early termination fees may apply as specified in the contract.\n\n"
            "4. How to Request a Refund: Send an email to support@cloudsync.com with your "
            "account ID and reason for cancellation. Include 'REFUND REQUEST' in the subject line.\n\n"
            "5. Processing Time: Approved refunds are processed within 5-10 business days "
            "and will appear on the original payment method."
        )
    },
    {
        "title": "Product Tiers & Pricing (2024)",
        "source": "sales/pricing-2024-q1.pdf",
        "last_updated": "January 2024",
        "content": (
            "CloudSync Solutions — Product Tiers & Pricing (Effective January 2024)\n\n"
            "STARTER PLAN — $29/month\n"
            "- Up to 5 users\n"
            "- 50 GB cloud storage\n"
            "- Email support (business hours)\n"
            "- Basic analytics dashboard\n\n"
            "PROFESSIONAL PLAN — $79/month\n"
            "- Up to 25 users\n"
            "- 500 GB cloud storage\n"
            "- Priority support with live chat\n"
            "- API access (1,000 requests/minute)\n"
            "- Advanced analytics and reporting\n\n"
            "ENTERPRISE PLAN — $199/month\n"
            "- Unlimited users\n"
            "- 5 TB cloud storage\n"
            "- Dedicated Customer Success Manager\n"
            "- SSO (SAML 2.0 and OIDC)\n"
            "- Custom integrations and onboarding\n"
            "- Unlimited API access\n\n"
            "ANNUAL BILLING DISCOUNT\n"
            "All plans receive a 20% discount when billed annually.\n"
            "- Starter Annual: $278/year (saves $70)\n"
            "- Professional Annual: $758/year (saves $190)\n"
            "- Enterprise Annual: $1,910/year (saves $478)\n\n"
            "ALL PLANS INCLUDE:\n"
            "- 99.9% uptime SLA\n"
            "- Daily automatic backups\n"
            "- HTTPS encryption for all data in transit\n"
            "- SOC 2 compliant infrastructure"
        )
    },
    {
        "title": "Support SLA (Service Level Agreement)",
        "source": "support/sla-2024.pdf",
        "last_updated": "January 2024",
        "content": (
            "CloudSync Solutions — Support Service Level Agreement (2024)\n\n"
            "STARTER PLAN SUPPORT:\n"
            "- Response time: Within 48 hours\n"
            "- Availability: Business hours only (Monday-Friday, 9am-5pm ET)\n"
            "- Channels: Email only\n"
            "- No dedicated account manager\n\n"
            "PROFESSIONAL PLAN SUPPORT:\n"
            "- Response time: Within 4 hours\n"
            "- Availability: Extended hours (Monday-Friday, 7am-9pm ET)\n"
            "- Channels: Email + Live chat\n"
            "- Access to senior support engineers\n\n"
            "ENTERPRISE PLAN SUPPORT:\n"
            "- Response time: Within 1 hour\n"
            "- Availability: 24/7/365\n"
            "- Channels: Email + Live chat + Phone + Dedicated Slack channel\n"
            "- Dedicated account manager assigned at onboarding\n\n"
            "CRITICAL ISSUE ESCALATION (System Down / Data Loss):\n"
            "- Enterprise: 15-minute response time, immediate escalation to engineering\n"
            "- Professional: 1-hour response time, escalation within 2 hours\n"
            "- Starter: Best-effort during business hours, no guaranteed escalation timeline"
        )
    },
    {
        "title": "Security & Compliance Overview",
        "source": "security/compliance-overview-2024.pdf",
        "last_updated": "February 2024",
        "content": (
            "CloudSync Solutions — Security & Compliance Overview\n\n"
            "CERTIFICATIONS & COMPLIANCE:\n"
            "- SOC 2 Type II certified (renewed annually, last audit: November 2023)\n"
            "- GDPR compliant — Data Processing Agreement (DPA) available upon request\n"
            "- CCPA compliant for California users\n\n"
            "DATA ENCRYPTION:\n"
            "- Data at rest: AES-256 encryption\n"
            "- Data in transit: TLS 1.3\n"
            "- Database backups: Encrypted with separate key management\n\n"
            "SECURITY TESTING:\n"
            "- Annual third-party penetration testing by CrowdShield Security\n"
            "- Continuous automated vulnerability scanning\n"
            "- Bug bounty program for external researchers\n\n"
            "DATA RESIDENCY OPTIONS:\n"
            "- United States (Virginia)\n"
            "- European Union (Frankfurt)\n"
            "- Asia-Pacific (Singapore)\n\n"
            "ACCESS CONTROLS:\n"
            "- MFA required for all admin accounts\n"
            "- Role-based access control (RBAC) with audit logging\n"
            "- SSO integration available on Enterprise plan (SAML 2.0 and OIDC)"
        )
    },
    {
        "title": "Employee Handbook — Remote Work Policy",
        "source": "hr/remote-work-policy-2024.pdf",
        "last_updated": "January 2024",
        "content": (
            "CloudSync Solutions — Remote Work Policy (Employee Handbook, Section 4.2)\n\n"
            "HYBRID WORK MODEL:\n"
            "CloudSync operates on a hybrid model. All employees are required to be "
            "in-office a minimum of 2 days per week: Tuesday and Thursday.\n\n"
            "HOME OFFICE EQUIPMENT:\n"
            "- One-time home office stipend: $1,500\n"
            "- Covers: desk, chair, monitor, peripherals\n"
            "- Must be requested within 90 days of start date\n"
            "- Receipts required for reimbursement\n\n"
            "CORE COLLABORATION HOURS:\n"
            "- All employees must be available 10:00 AM – 3:00 PM local time\n"
            "- Outside core hours, flexible scheduling is permitted\n"
            "- Manager approval needed for exceptions\n\n"
            "PAID TIME OFF:\n"
            "- Unlimited PTO with manager approval\n"
            "- Minimum 15 days per year strongly encouraged\n"
            "- Consecutive days exceeding 10 require VP approval\n\n"
            "TEAM EVENTS:\n"
            "- Quarterly in-person team events (company-funded travel and accommodation)\n"
            "- Annual company-wide offsite (location rotates annually)\n"
            "- Monthly virtual social events for remote employees"
        )
    },
    {
        "title": "Product FAQ",
        "source": "support/product-faq-2024.pdf",
        "last_updated": "March 2024",
        "content": (
            "CloudSync Solutions — Frequently Asked Questions\n\n"
            "Q: Can I upgrade my plan mid-billing-cycle?\n"
            "A: Yes! Upgrades take effect immediately. You'll be charged a pro-rated "
            "amount for the remainder of your current billing cycle.\n\n"
            "Q: Can I downgrade my plan?\n"
            "A: Yes. Downgrades take effect at the start of your next billing cycle. "
            "You'll retain access to higher-tier features until then.\n\n"
            "Q: What are the API rate limits?\n"
            "A: Professional plan: 1,000 requests per minute. Enterprise plan: Unlimited. "
            "Starter plan does not include API access.\n\n"
            "Q: How do I export my data?\n"
            "A: Go to Settings > Data > Export. Available formats: CSV and JSON. "
            "Exports include all records, attachments, and audit logs.\n\n"
            "Q: Is Single Sign-On (SSO) available?\n"
            "A: SSO is available on the Enterprise plan only. We support SAML 2.0 "
            "and OpenID Connect (OIDC). Contact your CSM for setup.\n\n"
            "Q: Is there a mobile app?\n"
            "A: Yes! CloudSync is available on iOS and Android. The mobile app syncs "
            "with your desktop account in real-time. Download from the App Store or Google Play.\n\n"
            "Q: Do you offer a free trial?\n"
            "A: Yes, all plans come with a 14-day free trial. No credit card required. "
            "Your trial automatically starts on the Professional plan so you can test all features."
        )
    },
    {
        "title": "Product Tiers & Pricing (2022 — OUTDATED)",
        "source": "archive/pricing-2022-legacy.pdf",
        "last_updated": "January 2022",
        "content": (
            "CloudSync Solutions — Product Tiers & Pricing\n"
            "Last updated: January 2022\n"
            "⚠️ THIS DOCUMENT IS OUTDATED — See pricing-2024-q1.pdf for current pricing\n\n"
            "STARTER PLAN — $19/month\n"
            "- Up to 3 users\n"
            "- 20 GB cloud storage\n"
            "- Email support\n\n"
            "PROFESSIONAL PLAN — $49/month\n"
            "- Up to 15 users\n"
            "- 200 GB cloud storage\n"
            "- Priority support\n"
            "- API access\n\n"
            "ENTERPRISE PLAN — $149/month\n"
            "- Unlimited users\n"
            "- 2 TB cloud storage\n"
            "- Dedicated support\n"
            "- Custom integrations\n\n"
            "Annual billing discount: 15% off all plans."
        )
    }
]

print("✅ 7 company documents loaded!")
print()
for i, doc in enumerate(company_documents):
    print(f"   📄 Doc {i+1}: {doc['title']}")
    print(f"      Source: {doc['source']}")
    print(f"      Updated: {doc['last_updated']}")
    print(f"      Length: {len(doc['content'])} chars")
    print()

---

## ✂️ Part 4: Document Chunking

### Why Chunk Documents?

When a customer asks *"What's the refund policy?"*, we don't need to read the entire security compliance document or the employee handbook. We only need the **relevant passage**.

**Chunking** breaks documents into smaller, overlapping pieces so our search tool can find the **most relevant section** instead of returning entire documents.

Why this matters:
- **Relevance**: Smaller chunks = more precise search results
- **Token efficiency**: We only send relevant text to Claude, not everything
- **Scalability**: Works even when you have thousands of documents

Our approach: Split each document into ~500-character chunks with overlap, keeping metadata (title, source, chunk index) attached to each piece.

In [ ]:
# ============================================
# DOCUMENT CHUNKING
# ============================================
# Split each document into overlapping chunks of ~500 characters.
# Each chunk keeps its metadata so we know where it came from.

def chunk_documents(documents, chunk_size=500, overlap=100):
    """
    Split documents into overlapping chunks.
    
    Args:
        documents: List of dicts with 'title', 'content', 'source' fields
        chunk_size: Target size of each chunk in characters
        overlap: Number of overlapping characters between chunks
    
    Returns:
        List of chunk dicts with 'text', 'title', 'source', 'chunk_index' fields
    """
    chunks = []
    
    for doc in documents:
        content = doc['content']
        title = doc['title']
        source = doc['source']
        last_updated = doc.get('last_updated', 'Unknown')
        
        # Split into chunks with overlap
        start = 0
        chunk_index = 0
        
        while start < len(content):
            end = start + chunk_size
            
            # Try to break at a paragraph or sentence boundary
            if end < len(content):
                # Look for a newline near the end
                newline_pos = content.rfind('\n', start + chunk_size - 100, end + 50)
                if newline_pos > start:
                    end = newline_pos
            
            chunk_text = content[start:end].strip()
            
            if chunk_text:  # Don't add empty chunks
                chunks.append({
                    'text': chunk_text,
                    'title': title,
                    'source': source,
                    'last_updated': last_updated,
                    'chunk_index': chunk_index
                })
                chunk_index += 1
            
            # Move forward, but overlap with previous chunk
            start = end - overlap if end < len(content) else len(content)
    
    return chunks


# Create chunks from our company documents
chunks = chunk_documents(company_documents)

print(f"✅ Created {len(chunks)} chunks from {len(company_documents)} documents!")
print()

# Show a few example chunks
for i, chunk in enumerate(chunks[:5]):
    print(f"   📝 Chunk {i}: [{chunk['title']}] (part {chunk['chunk_index']})")
    print(f"      Source: {chunk['source']}")
    print(f"      Updated: {chunk['last_updated']}")
    print(f"      Preview: {chunk['text'][:100]}...")
    print()

print(f"   ... and {len(chunks) - 5} more chunks")

---

## 🔍 Part 5: Build the Search Tool

Now we need a way to **find the right chunks** for a given question. This is the **retrieval** step in Retrieval-Augmented Generation.

In production systems, this would use **vector embeddings** and a vector database (like Pinecone, Weaviate, or ChromaDB). For class, we'll use **TF-IDF scoring** — it's transparent, teachable, and surprisingly effective for small document sets.

### 🔍 Building Our Search Tool — TF-IDF Scoring

Our search function uses **TF-IDF** (Term Frequency × Inverse Document Frequency) — the same algorithm behind classic search engines.

- **Term Frequency (TF):** How often a word appears in a chunk. More mentions = more relevant.
- **Inverse Document Frequency (IDF):** How rare a word is across ALL chunks. Rare words (like "refund") matter more than common words (like "the").
- **TF-IDF Score = TF × IDF** — ranks chunks by how uniquely relevant they are to your query.

> **Note:** This is keyword-based search. Production RAG systems use **vector embeddings** (Session 10) for semantic understanding — finding documents that MEAN the same thing even if they use different words. We start with TF-IDF because it's transparent and teachable.

In [ ]:
# ============================================
# TF-IDF DOCUMENT SEARCH
# ============================================
# Scores each chunk by TF-IDF: how relevant AND how unique each
# query term is within the chunk, relative to ALL chunks.
# No sklearn needed — just Counter from collections and math.log.

from collections import Counter
import math

def search_documents(query, chunks, top_k=3):
    """
    Search company documents using TF-IDF scoring.
    
    TF  = term frequency  — how often a word appears in THIS chunk
    IDF = inverse document freq — how rare a word is across ALL chunks
    Score = sum of TF * IDF for every query term found in the chunk
    """
    # Remove common stop words that don't help with relevance
    stop_words = {
        'the', 'a', 'an', 'is', 'are', 'was', 'were', 'what', 'how',
        'do', 'does', 'i', 'my', 'our', 'their', 'for', 'to', 'and',
        'or', 'of', 'in', 'on', 'at', 'can', 'about', 'it', 'be',
        'that', 'this', 'with', 'from', 'not', 'but', 'if', 'when',
        'there', 'which', 'each', 'have', 'has', 'had', 'will', 'would'
    }

    # Tokenize the query
    query_terms = [w for w in query.lower().split() if w not in stop_words]
    if not query_terms:
        return []

    # Pre-compute: for each term, how many chunks contain it? (for IDF)
    num_chunks = len(chunks)
    doc_freq = Counter()                       # term → number of chunks containing it
    chunk_word_counts = []                     # one Counter per chunk

    for chunk in chunks:
        words = chunk['text'].lower().split()
        word_count = Counter(words)
        chunk_word_counts.append(word_count)
        # Count each unique term once per chunk
        for term in set(words):
            doc_freq[term] += 1

    # Score every chunk
    scored = []
    for idx, chunk in enumerate(chunks):
        word_count = chunk_word_counts[idx]
        total_words = sum(word_count.values()) or 1   # avoid division by zero

        score = 0.0
        for term in query_terms:
            tf  = word_count[term] / total_words                               # term frequency
            idf = math.log((num_chunks + 1) / (doc_freq.get(term, 0) + 1))    # inverse doc freq (smoothed)
            score += tf * idf

        if score > 0:
            scored.append((round(score, 4), chunk))

    # Sort by score (highest first)
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


print("✅ search_documents() defined — TF-IDF scoring!")
print("   TF  = how often a query word appears in a chunk")
print("   IDF = how rare that word is across ALL chunks")
print("   Score = sum(TF × IDF) for every matching query term")

In [ ]:
# ============================================
# TEST THE SEARCH TOOL STANDALONE
# ============================================
# Before wiring it into the agent, let's make sure TF-IDF scoring works!
# Notice how rare, specific terms score higher than common ones.

test_queries = [
    "refund policy annual plan",
    "enterprise pricing cost",
    "data encryption security",
    "remote work office policy",
    "API rate limits"
]

for query in test_queries:
    print(f"\n🔍 Query: \"{query}\"")
    print(f"   {'Rank':<5} {'Score':<10} {'Document':<45} {'Chunk Preview'}")
    print(f"   {'─'*5} {'─'*10} {'─'*45} {'─'*40}")
    results = search_documents(query, chunks)
    for rank, (score, chunk) in enumerate(results, 1):
        title = chunk['title'][:43]
        preview = chunk['text'][:38].replace('\n', ' ')
        print(f"   {rank:<5} {score:<10.4f} {title:<45} \"{preview}...\"")
    print()

---

## 🎯 Part 6: Three Levels of Knowledge

Let's compare the **same company question** answered three different ways:

| Level | Approach | Pros | Cons |
|-------|----------|------|------|
| **Level 0** | No documents | Fast | Hallucinates everything |
| **Level 1** | Stuff ALL docs into prompt | Accurate | Doesn't scale (token limit!) |
| **Level 2** | RAG — search, then answer | Accurate + scalable | Needs good search |

**Question:** *"What is CloudSync's refund policy for annual plans?"*

In [ ]:
# ============================================
# LEVEL 0: No documents — Claude hallucinates
# ============================================
# We already saw this above, but let's run it again for direct comparison.

COMPANY_QUESTION = "What is CloudSync Solutions' refund policy for annual plans?"

print("=" * 60)
print("💤 LEVEL 0: No Documents (hallucination mode)")
print("=" * 60)

response_l0 = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=500,
    messages=[{"role": "user", "content": COMPANY_QUESTION}]
)

print(response_l0.content[0].text)
print("\n❌ Problem: Every detail above is MADE UP.")

In [ ]:
# ============================================
# LEVEL 1: Stuff ALL documents into the system prompt
# ============================================
# This works but doesn't scale. Watch the token count!

# Build a massive system prompt with ALL documents
all_docs_text = "You are a CloudSync Solutions support agent. Answer ONLY based on these documents:\n\n"
for doc in company_documents:
    all_docs_text += f"--- {doc['title']} (Source: {doc['source']}, Updated: {doc['last_updated']}) ---\n"
    all_docs_text += doc['content'] + "\n\n"

all_docs_text += (
    "\nIMPORTANT: Only answer based on the documents above. "
    "If the answer is not in the documents, say 'I don't have information on that.'"
)

print("=" * 60)
print("📦 LEVEL 1: Stuff ALL Documents Into Prompt")
print("=" * 60)
print(f"   System prompt size: {len(all_docs_text):,} characters")
print(f"   Estimated tokens: ~{len(all_docs_text) // 4:,}")
print()

response_l1 = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=500,
    system=all_docs_text,
    messages=[{"role": "user", "content": COMPANY_QUESTION}]
)

print(response_l1.content[0].text)
print(f"\n✅ Accurate! But we sent {len(all_docs_text):,} chars to answer one question.")
print("   With 7 docs this is fine. With 7,000 docs? Impossible.")
print("   This approach hits the token limit fast and costs more per query.")

In [ ]:
# ============================================
# LEVEL 2: RAG Agent — search relevant docs, then answer
# ============================================
# This is the production pattern: search first, read only what's relevant.

# Step 1: Define the search tool for Claude
search_tool = {
    "name": "search_company_docs",
    "description": (
        "Search CloudSync Solutions' company documents (policies, pricing, FAQ, "
        "support SLA, security docs, HR handbook). Returns the most relevant "
        "document passages for a query. Always search before answering "
        "company-specific questions."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query — use specific terms related to the question (e.g., 'refund policy annual', 'enterprise support SLA', 'data encryption')"
            }
        },
        "required": ["query"]
    }
}

# Step 2: The function that handles the tool call
def search_company_docs(query: str) -> str:
    """Search company documents and return formatted results."""
    results = search_documents(query, chunks, top_k=3)
    
    if not results:
        return json.dumps({"message": "No relevant documents found for this query. Try different search terms."})
    
    formatted_results = []
    for score, chunk in results:
        formatted_results.append({
            "relevance_score": round(score, 2),
            "document": chunk['title'],
            "source": chunk['source'],
            "last_updated": chunk['last_updated'],
            "content": chunk['text']
        })
    
    return json.dumps(formatted_results, indent=2)


print("✅ RAG search tool defined!")
print("   Name: search_company_docs")
print("   The agent will call this to find relevant document passages.")

In [ ]:
# ============================================
# THE RAG AGENT LOOP
# ============================================
# This is run_react_agent() from Session 6 with:
#   1. A RAG-focused system prompt
#   2. The search_company_docs tool
#   3. Same ReAct pattern: THINK → ACT → OBSERVE → REPEAT or CONCLUDE

def run_rag_agent(user_message: str, tools: list, tool_functions: dict, max_steps: int = 10):
    """
    Run a RAG knowledge agent using the ReAct framework.
    
    Searches company documents before answering. Cites sources.
    Says 'I don't know' when documents don't have the answer.
    """
    # The RAG system prompt — grounded in company documents
    system_prompt = (
        "You are a CloudSync Solutions knowledge agent using the ReAct framework.\n"
        "You answer questions about CloudSync's products, policies, and services "
        "using ONLY company documents.\n\n"
        "For every question:\n"
        "1. THINK: What specific information do I need? Which document likely has it?\n"
        "2. ACT: Search company documents with specific terms.\n"
        "3. OBSERVE: Did I find the relevant policy/info? Is it current?\n"
        "4. REPEAT or CONCLUDE with a sourced answer.\n\n"
        "CRITICAL RULES:\n"
        "- ALWAYS search documents before answering company-specific questions\n"
        "- ONLY state facts found in documents — never make up policies or numbers\n"
        "- If documents don't contain the answer, say: 'I don't have information on that in our current documents.'\n"
        "- Always cite which document your answer comes from\n"
        "- If you find conflicting information, flag it explicitly and note document dates\n"
        "- Pay attention to document dates — prefer the most recent version"
    )

    print(f"\n{'=' * 60}")
    print(f"💬 User: {user_message}")
    print(f"{'=' * 60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while step < max_steps:
        step += 1

        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        print(f"\n{'- ' * 30}")
        print(f"🔄 Step {step} | Stop reason: {response.stop_reason}")

        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text" and block.text.strip():
                    # THOUGHT step
                    print(f"\n   💭 THOUGHT:")
                    for line in block.text.strip().split('\n'):
                        print(f"      {line}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    # ACTION step
                    print(f"\n   🎯 ACTION: {tool_name}")
                    print(f"      Input: {json.dumps(tool_input, indent=2)[:200]}")

                    # Execute the tool
                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = json.dumps({"error": f"Unknown tool: {tool_name}"})

                    # OBSERVATION step
                    print(f"\n   👁️  OBSERVATION:")
                    result_preview = result[:400] + "..." if len(result) > 400 else result
                    for line in result_preview.split('\n')[:10]:
                        print(f"      {line}")
                    if len(result) > 400:
                        print(f"      [...truncated for display, full data sent to agent]")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            # FINAL ANSWER
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            print(f"\n{'=' * 60}")
            print(f"🤖 FINAL ANSWER:")
            print(f"{'=' * 60}")
            print(final_answer)
            print(f"\n✅ Done in {step} step(s)")

            return {"answer": final_answer, "messages": messages, "steps": step}

    print(f"\n⚠️ Max steps ({max_steps}) reached.")
    return {"answer": "Max steps reached.", "messages": messages, "steps": step}


print("✅ RAG agent loop defined!")
print("   Same ReAct pattern as Session 6, grounded in company documents.")

In [ ]:
# ============================================
# LEVEL 2: RAG Agent in action!
# ============================================
# Watch the agent SEARCH company docs, then answer from real data.
# Compare this to Level 0 (hallucination) and Level 1 (everything stuffed in).

rag_tools = [search_tool]
rag_functions = {"search_company_docs": search_company_docs}

print("=" * 60)
print("🧠 LEVEL 2: RAG Agent (search → read → answer)")
print("=" * 60)

result_l2 = run_rag_agent(COMPANY_QUESTION, rag_tools, rag_functions)

print("\n" + "⭐ " * 20)
print("COMPARISON:")
print("  Level 0: Hallucinated everything — sounded real but was fiction")
print(f"  Level 1: Correct, but sent {len(all_docs_text):,} chars to Claude")
print(f"  Level 2: Correct, searched only relevant chunks, cited sources")
print("\n  Level 2 scales to 7,000 documents. Level 1 does not.")

### 📊 Three Approaches Compared

| Approach | Accuracy | Citations | Handles Conflicts | Admits Uncertainty |
|----------|----------|-----------|-------------------|-------------------|
| 🔴 No RAG (pure LLM) | ❌ Hallucinates | ❌ None | ❌ N/A | ❌ Never |
| 🟡 Simple Context (dump all docs) | ⚠️ Usually correct | ⚠️ Sometimes | ⚠️ Misses some | ⚠️ Inconsistent |
| 🟢 RAG Agent (search + retrieve) | ✅ Grounded | ✅ Always | ✅ Detects & flags | ✅ Says "I don't know" |

**The progression:**
- **No RAG** = Confident and wrong --> dangerous
- **Simple Context** = Often right but can't scale --> limited
- **RAG Agent** = Right, cited, and honest --> production-ready

---

## 🧪 Part 7: Test the RAG Agent

Let's test with several different company questions. Watch how the agent:
- Searches for the right documents
- Cites its sources
- Handles questions that require combining information from multiple docs

### 🔍 What to Watch For — Test 1 (Clear Answer in Docs)

Before running this test, here's what a good RAG agent should do:

| Criterion | What to Look For | ✅ Good | ❌ Bad |
|-----------|-----------------|---------|--------|
| Right Document | Does it find the refund policy? | Uses Refund & Cancellation Policy | Uses random doc or none |
| Right Answer | Is the answer factually correct? | "30-day money-back guarantee" | "60-day refund" or made-up numbers |
| Citations | Does it cite its sources? | "According to refund-cancellation-v3..." | No source mentioned |
| Specificity | Does it include key details? | Mentions email to support@cloudsync.com | Vague or generic answer |

In [ ]:
# ============================================
# TEST 1: Clear answer in docs
# ============================================
# The refund policy is clearly stated in Document 1.
# The agent should find it and cite it.

test1 = run_rag_agent(
    "What is the refund policy for annual plans?",
    rag_tools,
    rag_functions
)

### 🔍 What to Watch For — Test 2 (Cross-Document Comparison)

Before running this test, here's what a good RAG agent should do when it needs info from multiple documents:

| Criterion | What to Look For | ✅ Good | ❌ Bad |
|-----------|-----------------|---------|--------|
| Right Documents | Does it find both pricing AND SLA docs? | Retrieves from pricing + support SLA | Only uses one document |
| Multiple Searches | Does the agent search more than once? | Searches "Professional plan" then "Enterprise support" | Single vague search |
| Synthesis | Does it compare the two plans clearly? | Side-by-side: response times, channels, hours | Just dumps raw text from docs |
| Accuracy | Are the specific details correct? | Professional: 4hr response, M-F 7am-9pm | Wrong response times or hours |

In [ ]:
# ============================================
# TEST 2: Requires comparing across documents
# ============================================
# This needs info from the Support SLA doc AND the Pricing doc.
# Watch if the agent searches more than once.

test2 = run_rag_agent(
    "What's the difference between Professional and Enterprise support?",
    rag_tools,
    rag_functions
)

### 🔍 What to Watch For — Test 3 (Question NOT in Documents)

Before running this test, here's how a good RAG agent should handle a question with no answer in the documents:

| Criterion | What to Look For | ✅ Good | ❌ Bad |
|-----------|-----------------|---------|--------|
| Right Behavior | Does it admit the info isn't available? | "I don't have information on HIPAA" | Makes up a HIPAA compliance policy |
| Search Attempt | Does it search before concluding? | Searches "HIPAA compliance" first | Answers immediately without searching |
| Honesty | Does it avoid guessing? | "Our documents don't cover HIPAA" | "CloudSync is HIPAA compliant" (fabricated) |
| Helpfulness | Does it suggest next steps? | "Contact sales for compliance questions" | Leaves the user with nothing |

In [ ]:
# ============================================
# TEST 3: NOT in the documents
# ============================================
# HIPAA compliance is NOT mentioned anywhere in our docs.
# A good RAG agent should say "I don't have information on that."
# A bad one will hallucinate a HIPAA policy.

test3 = run_rag_agent(
    "Does CloudSync support HIPAA compliance?",
    rag_tools,
    rag_functions
)

print("\n" + "🎯 " * 20)
print("DEBRIEF: Did the agent admit it couldn't find HIPAA info?")
print("Or did it make up a HIPAA compliance policy? Check the answer above.")

### 🔍 What to Watch For — Test 4 (Outdated Document Conflict)

Before running this test, here's what a good RAG agent should do when documents contradict each other:

| Criterion | What to Look For | ✅ Good | ❌ Bad |
|-----------|-----------------|---------|--------|
| Right Documents | Does it find BOTH pricing docs? | Retrieves 2024 AND 2022 pricing | Only finds one version |
| Conflict Detection | Does it flag the price discrepancy? | "2024 doc says $29, 2022 doc says $19" | Silently picks one price |
| Date Awareness | Does it prefer the newer document? | Uses 2024 pricing as current | Quotes 2022 pricing as current |
| Transparency | Does it warn about the outdated doc? | "Note: 2022 pricing is outdated" | Blends old and new prices together |

In [ ]:
# ============================================
# TEST 4: Potential conflict with outdated doc
# ============================================
# We have TWO pricing docs: 2024 ($29/mo) and 2022 ($19/mo).
# Does the agent find both? Does it flag the conflict?

test4 = run_rag_agent(
    "How much does the Starter plan cost?",
    rag_tools,
    rag_functions
)

print("\n" + "🎯 " * 20)
print("DEBRIEF: Did the agent find both pricing documents?")
print("Did it flag the conflict between 2024 ($29/mo) and 2022 ($19/mo)?")
print("Did it prefer the more recent document?")

---

## ⚠️ Part 8: Failure Modes — When RAG Goes Wrong

RAG isn't magic. It can fail in predictable ways. Understanding these failure modes is **critical** for deploying AI agents in business.

### The Three Failure Modes:

| Failure Mode | What Happens | Real-World Risk |
|---|---|---|
| **Outdated data** | Agent quotes old pricing, policies | Customer gets wrong price, legal liability |
| **Unanswerable question** | Agent makes up an answer | Company bound by fabricated policy |
| **Cross-document reasoning** | Agent can't combine info from multiple docs | Incomplete or wrong answer |

In [ ]:
# ============================================
# FAILURE MODE 1: Outdated Data
# ============================================
# The 2022 pricing doc says $19/mo. The 2024 doc says $29/mo.
# The agent might find BOTH. Does it flag the conflict?

print("⚠️ FAILURE TEST 1: Outdated Data")
print("   We have pricing from 2022 AND 2024. Which does the agent use?")
print()

fail1 = run_rag_agent(
    "What is the current pricing for all CloudSync plans? "
    "I need exact monthly costs.",
    rag_tools,
    rag_functions
)

print("\n" + "🔍 " * 20)
print("ANALYSIS: Check if the agent...")
print("  1. Found both pricing documents (2022 and 2024)")
print("  2. Flagged the conflict between old and new prices")
print("  3. Preferred the 2024 (current) document")
print("  4. Warned about the outdated 2022 document")

In [ ]:
# ============================================
# FAILURE MODE 2: Unanswerable Question
# ============================================
# Cryptocurrency payments are NOT mentioned in ANY document.
# The agent should say "I don't have information on that."

print("⚠️ FAILURE TEST 2: Unanswerable Question")
print("   Cryptocurrency payments are not in any document.")
print()

fail2 = run_rag_agent(
    "What is CloudSync's policy on cryptocurrency payments? "
    "Can I pay with Bitcoin?",
    rag_tools,
    rag_functions
)

print("\n" + "🔍 " * 20)
print("ANALYSIS: A GOOD agent says 'I don't have information on that.'")
print("  A BAD agent makes up a cryptocurrency policy.")
print("  Which did yours do?")

In [ ]:
# ============================================
# FAILURE MODE 3: Cross-Document Reasoning
# ============================================
# This requires combining info from:
#   - Support SLA (Professional: 1-hour critical response, M-F 7am-9pm)
#   - Possibly the pricing doc (what Professional includes)
# 2am Saturday is OUTSIDE Professional support hours!

print("⚠️ FAILURE TEST 3: Cross-Document Reasoning")
print("   Requires combining Support SLA + plan details.")
print()

fail3 = run_rag_agent(
    "If I'm on the Professional plan and have a critical outage at 2am "
    "on Saturday, what happens? How quickly will I get a response?",
    rag_tools,
    rag_functions
)

print("\n" + "🔍 " * 20)
print("ANALYSIS: The tricky part is that Professional support hours are")
print("  M-F 7am-9pm ET. A 2am Saturday outage is OUTSIDE those hours.")
print("  Did the agent catch this? Or did it just quote the 1-hour SLA?")

### ✈️ The Air Canada Case — Why RAG Accuracy Has Legal Consequences

In **2024**, Air Canada's chatbot told a customer named Jake Moffatt that he could get a **bereavement fare discount after booking** his flight. When Moffatt tried to claim the discount, Air Canada said the chatbot was wrong — their actual policy required applying *before* travel.

Moffatt took it to a **civil resolution tribunal**. Air Canada argued the chatbot was a "separate legal entity" responsible for its own statements.

**The tribunal ruled against Air Canada.** The chatbot's answer counted as a **binding offer** from the company. Air Canada had to pay Moffatt the difference.

### What This Means for RAG:

- Your agent's answers can create **legal obligations** for your company
- Outdated documents in the knowledge base = **liability**
- An agent that hallucinates policies is worse than no agent at all
- **"I don't know"** is not a failure — it's a **safety feature**

This is why we test failure modes. This is why we build agents that cite sources and flag uncertainty. This is why RAG accuracy matters.

---

## 🏆 Part 9: Enhanced Agent with Citations

Let's upgrade our system prompt to require **structured citations** in every answer. This makes the agent's responses auditable and trustworthy.

In [ ]:
# ============================================
# ENHANCED RAG AGENT WITH CITATIONS
# ============================================
# Same agent loop, upgraded system prompt for better output quality.

def run_rag_agent_with_citations(user_message: str, tools: list, tool_functions: dict, max_steps: int = 10):
    """
    RAG agent with enforced citation format.
    Requires [Source: Document Name] after each factual claim.
    """
    # Enhanced system prompt with citation requirements
    system_prompt = (
        "You are a CloudSync Solutions knowledge agent using the ReAct framework.\n"
        "You answer questions about CloudSync's products, policies, and services "
        "using ONLY company documents.\n\n"
        "For every question:\n"
        "1. THINK: What specific information do I need? Which document likely has it?\n"
        "2. ACT: Search company documents with specific terms.\n"
        "3. OBSERVE: Did I find the relevant policy/info? Is it current?\n"
        "4. REPEAT or CONCLUDE with a sourced answer.\n\n"
        "CITATION RULES (MANDATORY):\n"
        "When answering, ALWAYS use this format:\n"
        "- State the answer clearly\n"
        "- Add [Source: Document Name] after each factual claim\n"
        "- If confidence is low, explicitly say so\n"
        "- If you find conflicting info, show BOTH versions with dates\n\n"
        "Example format:\n"
        "'The Starter plan costs $29/month [Source: Product Tiers & Pricing (2024)]. "
        "Note: An older document from 2022 lists the price as $19/month "
        "[Source: Product Tiers & Pricing (2022 — OUTDATED)]. "
        "The 2024 pricing is current.'\n\n"
        "CRITICAL RULES:\n"
        "- ALWAYS search documents before answering company-specific questions\n"
        "- ONLY state facts found in documents — never make up policies or numbers\n"
        "- If documents don't contain the answer, say: 'I don't have information on that in our current documents.'\n"
        "- If you find conflicting information, flag it explicitly and note document dates\n"
        "- Pay attention to document dates — prefer the most recent version"
    )

    print(f"\n{'=' * 60}")
    print(f"💬 User: {user_message}")
    print(f"{'=' * 60}")

    messages = [{"role": "user", "content": user_message}]
    step = 0

    while step < max_steps:
        step += 1

        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages
        )

        print(f"\n{'- ' * 30}")
        print(f"🔄 Step {step} | Stop reason: {response.stop_reason}")

        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "text" and block.text.strip():
                    print(f"\n   💭 THOUGHT:")
                    for line in block.text.strip().split('\n'):
                        print(f"      {line}")

                elif block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input

                    print(f"\n   🎯 ACTION: {tool_name}")
                    print(f"      Input: {json.dumps(tool_input, indent=2)[:200]}")

                    if tool_name in tool_functions:
                        result = tool_functions[tool_name](**tool_input)
                    else:
                        result = json.dumps({"error": f"Unknown tool: {tool_name}"})

                    print(f"\n   👁️  OBSERVATION:")
                    result_preview = result[:400] + "..." if len(result) > 400 else result
                    for line in result_preview.split('\n')[:10]:
                        print(f"      {line}")
                    if len(result) > 400:
                        print(f"      [...truncated for display]")

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer += block.text

            print(f"\n{'=' * 60}")
            print(f"🤖 FINAL ANSWER:")
            print(f"{'=' * 60}")
            print(final_answer)
            print(f"\n✅ Done in {step} step(s)")

            return {"answer": final_answer, "messages": messages, "steps": step}

    print(f"\n⚠️ Max steps ({max_steps}) reached.")
    return {"answer": "Max steps reached.", "messages": messages, "steps": step}


print("✅ Enhanced RAG agent with citations defined!")

In [ ]:
# ============================================
# COMPARE: Same question, with and without citation requirements
# ============================================

print("\n" + "📝 " * 20)
print("ENHANCED AGENT (with mandatory citations):")
print("📝 " * 20)

cite_result = run_rag_agent_with_citations(
    "What are the key differences between the Starter and Enterprise plans? "
    "Compare pricing, storage, support, and security features.",
    rag_tools,
    rag_functions
)

print("\n" + "⭐ " * 20)
print("Notice the [Source: ...] citations after each claim.")
print("This makes the answer auditable — you can verify every fact.")
print("This is the standard for production RAG agents.")

In [ ]:
# ============================================
# BONUS: Test the citation agent with a tricky question
# ============================================
# This should trigger the "I don't know" response with citations
# for the parts it CAN find, and honesty about what it can't.

cite_result_2 = run_rag_agent_with_citations(
    "I'm evaluating CloudSync for a healthcare company. We need HIPAA compliance, "
    "data encryption, and 24/7 support. Can CloudSync meet these requirements?",
    rag_tools,
    rag_functions
)

print("\n" + "🎯 " * 20)
print("ANALYSIS: The agent should...")
print("  ✅ Cite encryption details from Security & Compliance doc")
print("  ✅ Cite 24/7 support from SLA doc (Enterprise plan)")
print("  ❌ Admit HIPAA is NOT in the documents")
print("  Did it handle all three correctly?")

---

## 🎨 Part 10: Build Your Own Knowledge Agent (Challenge)

You've built a RAG agent for CloudSync Solutions. Now it's your turn: create a knowledge agent for a **domain you care about**.

### Ideas:
- **Your internship/job**: Company policies, onboarding docs, product info
- **A company you admire**: Public docs, FAQ pages, support articles
- **Your university**: Course catalog, financial aid policies, housing rules
- **A hobby/interest**: Rules of a sport, game strategy guides, recipe collections

### Template below — fill in YOUR documents:

### 💡 Domain Ideas for Your RAG Agent

**Option 1: University Course Catalog Agent**
- Load 5-10 course descriptions from Suffolk's catalog
- Test: "What are the prerequisites for ISOM 260?" / "Which courses cover machine learning?"
- Edge case: What if two departments list similar courses with different prereqs?

**Option 2: Restaurant Menu & Policy Agent**
- Create 5 documents: menu, allergen info, reservation policy, hours, catering options
- Test: "Do you have gluten-free options?" / "Can I book a party of 20?"
- Edge case: What if the menu says "seasonal items" but doesn't list current season?

**Option 3: Startup Product FAQ Agent**
- Create docs: pricing page, feature list, API docs, competitor comparison, changelog
- Test: "How does your API rate limiting work?" / "What's different from Competitor X?"
- Edge case: What if the changelog contradicts the current feature list?

### ✅ Testing Checklist

For each domain, test these scenarios:
- [ ] Question answerable from one document --> should cite that document
- [ ] Question requiring info from multiple documents --> should synthesize
- [ ] Question with no answer in documents --> should say "I don't know"
- [ ] Question where documents contradict --> should flag the conflict
- [ ] Question about something that changed --> should use most recent document

### 🍕 Worked Example: Restaurant Agent

Here's a minimal working example with 3 documents to show you the pattern before you build your own:

In [ ]:
# ============================================
# WORKED EXAMPLE: Restaurant Knowledge Agent
# ============================================
# A minimal RAG agent with just 3 documents — enough to see the pattern.

restaurant_documents = [
    {
        "title": "Menu — Main Dishes",
        "source": "menu.pdf",
        "last_updated": "2024",
        "content": (
            "Our signature dishes include: Truffle Mushroom Risotto ($24) — creamy arborio rice "
            "with wild mushrooms and black truffle oil. Grilled Atlantic Salmon ($28) — with lemon "
            "herb butter, seasonal vegetables, and roasted potatoes. Classic Margherita Pizza ($16) "
            "— San Marzano tomatoes, fresh mozzarella, and basil on house-made dough. All entrees "
            "are served with a house salad or soup of the day."
        )
    },
    {
        "title": "Allergen Information",
        "source": "allergens.pdf",
        "last_updated": "2024",
        "content": (
            "We take allergies seriously. Our kitchen handles: nuts, dairy, gluten, shellfish, and "
            "soy. Gluten-free pasta and pizza dough available on request (+$3). Vegan cheese "
            "substitute available for any pizza (+$2). Please inform your server of any allergies "
            "before ordering — our chef can modify most dishes. Full allergen matrix available on "
            "our website."
        )
    },
    {
        "title": "Reservations & Events",
        "source": "reservations.pdf",
        "last_updated": "2024",
        "content": (
            "Reservations recommended for parties of 4+. Walk-ins welcome but wait times may apply "
            "on weekends. Private dining room seats up to 30 guests — booking requires $500 deposit. "
            "Catering available for off-site events, minimum order $800. Monday–Thursday: 11am–9pm, "
            "Friday–Saturday: 11am–11pm, Sunday: 10am–8pm (brunch served until 2pm)."
        )
    },
]

# Chunk and index the restaurant documents
restaurant_chunks = chunk_documents(restaurant_documents)
print(f"\u2705 Created {len(restaurant_chunks)} chunks from {len(restaurant_documents)} restaurant documents")

# Build the search function for the restaurant
def search_restaurant_docs(query: str) -> str:
    """Search restaurant documents."""
    results = search_documents(query, restaurant_chunks, top_k=3)
    if not results:
        return json.dumps({"message": "No relevant documents found."})
    formatted = []
    for score, chunk in results:
        formatted.append({
            "relevance_score": round(score, 2),
            "document": chunk['title'],
            "source": chunk['source'],
            "content": chunk['text']
        })
    return json.dumps(formatted, indent=2)

# Define the tool for Claude
restaurant_search_tool = {
    "name": "search_restaurant_docs",
    "description": (
        "Search the restaurant knowledge base including menu, allergen info, "
        "and reservation policies. Returns relevant passages for any query."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query about the restaurant"
            }
        },
        "required": ["query"]
    }
}

restaurant_tools = [restaurant_search_tool]
restaurant_functions = {"search_restaurant_docs": search_restaurant_docs}

# Sample questions to test your restaurant agent
test_questions = [
    "Do you have gluten-free options?",
    "Can I book a party of 20?",
    "What are your hours on Monday?"
]

# Try it out!
for q in test_questions:
    print(f"\nQ: {q}")
    result = run_rag_agent(q, restaurant_tools, restaurant_functions)
    print(f"A: {result['answer']}\n")
    print("-" * 40)

In [ ]:
# ============================================
# YOUR TURN: Build a Knowledge Agent for YOUR Domain
# ============================================

# Step 1: Define your company/domain documents
# Replace these with REAL content from a domain you care about!

my_documents = [
    {
        "title": "Your Document Title 1",         # <-- Change this
        "source": "source/filename.pdf",           # <-- Change this
        "last_updated": "2024",                    # <-- Change this
        "content": (
            "Paste your document content here. "   # <-- Replace with real content!
            "This could be a company policy, FAQ, product description, "
            "or any text your agent should know about."
        )
    },
    {
        "title": "Your Document Title 2",         # <-- Change this
        "source": "source/filename2.pdf",          # <-- Change this
        "last_updated": "2024",                    # <-- Change this
        "content": (
            "Another document. The more documents you add, "
            "the more useful your agent becomes. "
            "Aim for at least 5 documents."
        )
    },
    {
        "title": "Your Document Title 3",         # <-- Change this
        "source": "source/filename3.pdf",          # <-- Change this
        "last_updated": "2024",                    # <-- Change this
        "content": (
            "Add as many documents as you want! "
            "Copy content from real sources."
        )
    },
    # Add more documents here...
    # {
    #     "title": "Document 4",
    #     "source": "...",
    #     "last_updated": "...",
    #     "content": "..."
    # },
]

# Step 2: Chunk your documents
my_chunks = chunk_documents(my_documents)
print(f"✅ Created {len(my_chunks)} chunks from {len(my_documents)} documents")

# Step 3: Create your search function
def search_my_docs(query: str) -> str:
    """Search your custom documents."""
    results = search_documents(query, my_chunks, top_k=3)
    if not results:
        return json.dumps({"message": "No relevant documents found."})
    formatted = []
    for score, chunk in results:
        formatted.append({
            "relevance_score": round(score, 2),
            "document": chunk['title'],
            "source": chunk['source'],
            "content": chunk['text']
        })
    return json.dumps(formatted, indent=2)

# Step 4: Define the tool for Claude
my_search_tool = {
    "name": "search_my_docs",
    "description": (
        "Search your custom knowledge base. "        # <-- Update this description
        "Returns relevant document passages for any query."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query with specific terms"
            }
        },
        "required": ["query"]
    }
}

my_rag_tools = [my_search_tool]
my_rag_functions = {"search_my_docs": search_my_docs}

# Step 5: Test your agent!
# Uncomment and modify these:
# run_rag_agent_with_citations("Your question 1", my_rag_tools, my_rag_functions)
# run_rag_agent_with_citations("Your question 2", my_rag_tools, my_rag_functions)
# run_rag_agent_with_citations("Your question 3", my_rag_tools, my_rag_functions)

print("\n🚀 Replace the placeholder documents above with YOUR content!")
print("   Then uncomment the test questions and run the agent.")

---

## 💭 Reflection: What You Built Today

### The Stack So Far

```
Session 4: Tools          → Claude can DO things (call functions)
Session 5: Real APIs      → Claude connects to the real world (Wikipedia, web, math)
Session 6: ReAct          → Claude THINKS before acting (reason + act loop)
Session 7: RAG (TODAY)    → Claude knows YOUR business (grounded in documents)
```

### What You Built vs. Production Systems

| What You Built Today | Production RAG Systems |
|---|---|
| 7 documents, keyword search | Millions of docs, vector embeddings |
| Simple word overlap scoring | Semantic similarity with embedding models |
| Documents in Python dicts | Vector databases (Pinecone, Weaviate, ChromaDB) |
| Manual document creation | Automated ingestion pipelines |
| Single agent with search tool | Multi-agent RAG with re-ranking, filtering |
| Runs in Colab for class | Runs 24/7, serves thousands of queries |

### But the Core Pattern Is Identical

```
1. User asks a question
2. Agent SEARCHES relevant documents
3. Agent READS the retrieved passages
4. Agent GENERATES an answer grounded in the documents
5. Agent CITES its sources
```

You built the same architecture. Production adds:
- **Better search** — vector embeddings understand meaning, not just keywords
- **More documents** — millions instead of seven
- **Re-ranking** — a second pass to find the BEST results
- **Guardrails** — automated checks before answers reach users
- **Feedback loops** — learn from user corrections

### 💡 The Business Insight

RAG is the **most deployed AI pattern in enterprise**. Every company building AI agents needs RAG:

- **Customer support bots** grounded in knowledge base articles
- **Internal search** across company wikis and documents
- **Legal research** searching through contracts and regulations
- **Medical assistants** referencing clinical guidelines
- **Sales enablement** pulling from product specs and competitive analysis

The hard part isn't the code (you just wrote it in ~100 lines). The hard part is:

1. **Data quality** — Are your documents accurate, current, and complete?
2. **Retrieval quality** — Does the search find the RIGHT document?
3. **Answer quality** — Does the agent interpret the document correctly?
4. **Trust** — When should users trust the agent vs. escalate to a human?

These are business decisions, not engineering decisions.

---

## 🚀 What's Next?

### Homework

1. **Build:** Create a RAG agent for a domain you care about — load 5+ documents (articles, policies, product pages) and build a search tool. Test with 10 questions.

2. **Break:** Deliberately feed your agent contradicting documents and outdated data. Document 3 failure modes and how you'd fix each one.

3. **Evaluate:** Score your agent's accuracy: For 10 test questions, record whether it retrieved the right document, gave a correct answer, and cited sources properly.

4. **Design:** Write a 1-page proposal for a knowledge agent at a real company. What documents would it need? Who would use it? What's the risk if it's wrong?

### Next Session: Multi-Agent Systems

One agent is useful. A **team of agents** is powerful. In Session 8, you'll build a researcher → analyst pipeline and learn how agents coordinate to solve complex problems.

```
Session 8 Preview:
  Agent 1 (Researcher) → searches documents and web
  Agent 2 (Analyst) → synthesizes findings into a report
  Agent 3 (Editor) → checks citations and quality
```

---

**ISOM 260: AI for Business** | Suffolk University | Session 7

🌐 [isom-260.vercel.app](https://isom-260.vercel.app)